In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# ============================================================
# 1) LOAD DATA
# ============================================================
"""
EXPLANATORY NOTES:
- In a production IRB workflow, this step is where you enforce:
  * schema checks (expected columns + types)
  * versioning (data snapshot ID)
  * audit trail (who ran it, when)
"""

data_path = Path("Bank_Loan_Approval.csv")  # for GitHub: store under data/
df = pd.read_csv(data_path)

# Rename to clean, code-friendly names (keep semantics unchanged)
df = df.rename(columns={"Avg_CC Score": "Avg_CC_Score", "ZIP.Code": "ZIP_Code"})

In [2]:
# ============================================================
# 2) DATA QUALITY CHECKS + TREATMENT
# ============================================================
"""
EXPLANATORY NOTES:
Basel/IRB models require strong data governance. Typical treatment includes:
- Missingness handling
- Reasonableness checks (age ranges, non-negative values)
- Capping/winsorization for extreme values
- Dummy variable enforcement (0/1)
"""

num_cols = [
    "Age","Experience","Income","Family","Avg_CC_Score","Education","Mortgage",
    "Securities.Account","CD.Account","Online","CreditCard"
]

# Coerce numeric (protects against stray strings)
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Missingness report (store for model risk documentation)
missing_rate = df.isna().mean().sort_values(ascending=False)

# Median imputation (simple + stable baseline)
for c in num_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())

# Reasonableness / plausibility constraints
df["Age"] = df["Age"].clip(18, 100)
df["Income"] = df["Income"].clip(lower=0)
df["Mortgage"] = df["Mortgage"].clip(lower=0)
df["Family"] = df["Family"].clip(lower=0, upper=20)
df["Education"] = df["Education"].clip(1, 3)

# Experience must be >=0 and not exceed adult working years proxy (Age - 18)
df["Experience"] = df["Experience"].clip(lower=0)
max_exp = (df["Age"] - 18).clip(lower=0)
df["Experience"] = np.minimum(df["Experience"], max_exp)

# Force dummies to 0/1
dummy_cols = ["Securities.Account","CD.Account","Online","CreditCard"]
for c in dummy_cols:
    df[c] = (df[c] >= 0.5).astype(int)

In [3]:
# ============================================================
# 3) FEATURE SET (drop ZIP from modeling)
# ============================================================
"""
EXPLANATORY NOTES:
- ZIP can behave like an identifier/proxy for geography and can be problematic
  (fair lending, stability, leakage).
- We keep ZIP for reporting, but exclude from PD proxy scoring.
"""

feature_cols = [
    "Age","Experience","Income","Family","Avg_CC_Score","Education","Mortgage",
    "Securities.Account","CD.Account","Online","CreditCard"
]
X = df[feature_cols].copy()

# Standardize numeric features for stable scoring
scale_cols = ["Age","Experience","Income","Family","Avg_CC_Score","Education","Mortgage"]
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[scale_cols] = scaler.fit_transform(X_scaled[scale_cols])

In [4]:
# ============================================================
# 4) PROXY PD MODEL (since no observed default flag exists)
# ============================================================
"""
EXPLANATORY NOTES:
- This dataset has no default/approval outcome column.
- So we build an explainable proxy PD score (scorecard-like),
  and map it to a probability via a logistic function.
- Coefficient directions follow common risk intuition:
    higher Income / higher Avg_CC_Score / assets (CD, Securities) => lower PD
    higher Mortgage / larger Family => higher PD
"""

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

coefs = pd.Series({
    "Age": -0.05,
    "Experience": -0.10,
    "Income": -0.35,
    "Family":  0.15,
    "Avg_CC_Score": -0.25,
    "Education": -0.12,
    "Mortgage": 0.30,
    "Securities.Account": -0.20,
    "CD.Account": -0.30,
    "Online": -0.03,
    "CreditCard": 0.02
})

score_no_intercept = (X_scaled * coefs[X_scaled.columns]).sum(axis=1)

# Calibrate intercept to get portfolio mean PD near a realistic target (e.g., ~5%)
target_mean_pd = 0.05
lo, hi = -10.0, 10.0
for _ in range(60):
    mid = (lo + hi) / 2
    m = sigmoid(score_no_intercept + mid).mean()
    if m > target_mean_pd:
        hi = mid
    else:
        lo = mid
intercept = (lo + hi) / 2

df["prob_def"] = sigmoid(score_no_intercept + intercept)

In [5]:
# ============================================================
# 5) POLICY LAYER: APPROVAL DECISION FROM PD CUTOFF
# ============================================================
"""
EXPLANATORY NOTES:
- Banks approve/decline based on risk appetite and strategy.
- A common illustrative cutoff is PD <= 5% for approval,
  but in real life this is optimized vs:
    approval rate, bad rate, margin, capital, and expected loss.
"""

pd_cutoff = 0.05
df["loan_approval"] = (df["prob_def"] <= pd_cutoff).astype(int)

In [6]:
# ============================================================
# 6) EAD, LGD, EL (transparent proxy assumptions)
# ============================================================
"""
EXPLANATORY NOTES:
Basel IRB definitions:
- EAD (Exposure at Default): drawn + CCF * undrawn
- LGD (Loss Given Default): % loss severity after recoveries/collateral
- EL (Expected Loss): PD * EAD * LGD

This dataset has no loan amount, limit, utilization, or recoveries.
So we define proxies suitable for a GitHub template:

EAD proxy:
- Offered loan amount = 30% of Income (Income assumed in $000)
- EAD = 0.30 * Income * 1000
- Apply "1 is proxy for approved": if declined, EAD = 0

LGD proxy:
- Base unsecured LGD = 60%
- If CD or Securities accounts exist => lower LGD (better recovery proxy) = 40%
- Mortgage leverage adds small LGD uplift (capped between 20% and 90%)
"""

# EAD proxy (approved-only exposure)
df["EAD"] = 0.30 * df["Income"] * 1000.0
df["EAD"] = df["EAD"] * df["loan_approval"]

# LGD proxy
base_lgd = 0.60
lgd = np.full(len(df), base_lgd, dtype=float)

has_assets = (df["CD.Account"] == 1) | (df["Securities.Account"] == 1)
lgd[has_assets] = 0.40

mort_adj = (df["Mortgage"] / (df["Mortgage"].quantile(0.75) + 1e-9)) * 0.05
df["LGD"] = np.clip(lgd + mort_adj, 0.20, 0.90)

# Expected Loss
df["EL"] = df["prob_def"] * df["EAD"] * df["LGD"]

In [7]:
# ============================================================
# 7) EXPORT FINAL RESULT
# ============================================================
"""
EXPLANATORY NOTES:
- Always export the scored dataset used for decisions:
  includes PD, approval, and loss metrics for auditability.
"""

out_path = Path("outputs/Bank_Loan_Approval_scored.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)

# Optional: quick portfolio summary (useful in README / model note)
summary = {
    "rows": len(df),
    "mean_PD": float(df["prob_def"].mean()),
    "median_PD": float(df["prob_def"].median()),
    "approval_rate_at_5pct_cutoff": float(df["loan_approval"].mean()),
    "avg_EAD_approved": float(df.loc[df["loan_approval"]==1, "EAD"].mean()),
    "avg_LGD_approved": float(df.loc[df["loan_approval"]==1, "LGD"].mean()),
    "portfolio_EL_total": float(df["EL"].sum()),
}
print(summary)
print(f"Saved: {out_path.resolve()}")

{'rows': 5000, 'mean_PD': 0.05000000000000001, 'median_PD': 0.0469048522331528, 'approval_rate_at_5pct_cutoff': 0.553, 'avg_EAD_approved': 28450.632911392404, 'avg_LGD_approved': 0.5794890158092066, 'portfolio_EL_total': 1275704.2904052255}
Saved: C:\Users\Omotayo.Owolabi\outputs\Bank_Loan_Approval_scored.csv
